# EDA — IMDb reviews

50,000 labelled movie reviews (balanced positive/negative). Goal of this notebook: understand data quality and text structure before modelling.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/IMDB_reviews.csv")
df.shape

## Data quality

Checking nulls and duplicates. The duplicate check matters: identical reviews that land in both train and test would leak and inflate metrics.

In [ ]:
print("nulls:\n", df.isnull().sum())
print("exact duplicate rows:", df.duplicated().sum())
print("duplicate review texts:", df["review"].duplicated().sum())

There are 418 duplicated reviews. I drop them before the train/test split in the modelling notebook to avoid leakage.

In [ ]:
ax = df["sentiment"].value_counts().plot.bar(color=["#4C72B0", "#DD8452"])
ax.set(title="Class balance", xlabel="sentiment", ylabel="count")
plt.tight_layout()

## Review length

Length in characters, split by sentiment — checking for structural differences and outliers.

In [ ]:
df["length"] = df["review"].str.len()
df["length"].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["length"], bins=50, ax=axes[0])
axes[0].set(title="Review length", xlabel="characters")
sns.boxplot(data=df, x="sentiment", y="length", ax=axes[1])
axes[1].set(title="Length by sentiment")
plt.tight_layout()

Both classes have similar length distributions (median ~970 chars, a long right tail). Length alone is not a useful signal.

## Frequent vocabulary

Word clouds per class, using the same cleaning that feeds the model.

In [ ]:
from wordcloud import WordCloud
from sentiment_classifier.preprocessing import clean_text

cleaned = df["review"].map(clean_text)
pos = " ".join(cleaned[df["sentiment"] == "positive"])
neg = " ".join(cleaned[df["sentiment"] == "negative"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, text, title in [(axes[0], pos, "Positive"), (axes[1], neg, "Negative")]:
    wc = WordCloud(width=800, height=400, background_color="white",
                   stopwords={"movie", "film", "br"}).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(title)
plt.tight_layout()

## Takeaways

- Balanced, no nulls, 418 duplicate reviews to drop.
- Length carries no class signal; the vocabulary does.
- A TF-IDF + linear model is a strong, cheap baseline — see `02_modeling`.